In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTETomek
from collections import Counter

# Hypothèse de Modélisation : Prédiction de l'État des Planteras

## Hypothèse Principale

**L'état des planteras (`estado_plantera`) peut être prédit avec précision à partir des caractéristiques physiques des arbres, de leur localisation géographique et des conditions environnementales.** Cette hypothèse suppose que des variables comme la hauteur des arbres, le diamètre, l'espèce botanique, la commune et les caractéristiques de la plantera contiennent des signaux prédictifs suffisants pour déterminer si une plantera est en "bon état", "état régulier" ou "mauvais état".

## 🛠️ Boîte à Outils Statistique

### **Gestion du Déséquilibre des Classes**
- **SMOTE** : Technique de sur-échantillonnage qui génère des instances synthétiques des classes minoritaires pour équilibrer le dataset
- **Équilibrage par Poids** : Attribution de poids différents aux classes pendant l'entraînement pour penaliser davantage les erreurs sur les classes sous-représentées

### **Préparation des Données**
- **Encodage par Hachage** : Transformation des variables catégorielles textuelles (noms scientifiques, communes) en représentations numériques compatibles avec les algorithmes d'apprentissage automatique

### **Optimisation du Modèle**
- **GridSearchCV** : Recherche systématique par validation croisée des meilleurs hyperparamètres pour maximiser les performances du modèle Random Forest

### **Feature Engineering**
- **Création de variables dérivées** : Ratio hauteur/diamètre, estimation du volume
- **Normalisation** : Standardisation des features numériques pour améliorer la convergence


In [3]:
# Chargement et préparation des données
df = pd.read_csv('../data/datos_arbolado_clean.csv')
df_clean = df.dropna(subset=['altura_arbol', 'diametro_altura_pecho', 'comuna', 'nombre_cientifico', 'ubicacion_plantera','ancho_acera','nivel_plantera','estado_plantera'])

# Identifier et supprimer les classes rares
class_counts = df_clean['estado_plantera'].value_counts()
rare_classes = class_counts[class_counts <= 2].index
df_filtered = df_clean[~df_clean['estado_plantera'].isin(rare_classes)]

print("=== DISTRIBUTION DES CLASSES ===")
print(df_filtered['estado_plantera'].value_counts())
print("\n=== RAPPORT DE DÉSÉQUILIBRE ===")
majority_class = class_counts.max()
minority_class = class_counts.min()
print(f"Ratio classe majoritaire/minoritaire: {majority_class/minority_class:.2f}")
print()

=== DISTRIBUTION DES CLASSES ===
estado_plantera
2.0    302270
3.0     35058
1.0       167
Name: count, dtype: int64

=== RAPPORT DE DÉSÉQUILIBRE ===
Ratio classe majoritaire/minoritaire: 151135.00



In [4]:
# FEATURE ENGINEERING AVANCÉ
X = df_filtered[['altura_arbol', 'diametro_altura_pecho', 'comuna', 'nombre_cientifico','ancho_acera','nivel_plantera','ubicacion_plantera']]
y = df_filtered['estado_plantera']

# Encodage des variables catégorielles
le_nombre = LabelEncoder()
le_ubicacion = LabelEncoder()

X_encoded = X.copy()
X_encoded['nombre_cientifico_encoded'] = le_nombre.fit_transform(X['nombre_cientifico'])
X_encoded['ubicacion_plantera_encoded'] = le_ubicacion.fit_transform(X['ubicacion_plantera'])

# Feature engineering supplémentaire
X_encoded['ratio_altura_diametro'] = X_encoded['altura_arbol'] / (X_encoded['diametro_altura_pecho'] + 1e-5)
X_encoded['volume_estimate'] = X_encoded['altura_arbol'] * (X_encoded['diametro_altura_pecho'] ** 2)
X_encoded['comuna_encoded'] = LabelEncoder().fit_transform(X_encoded['comuna'])

# Supprimer les colonnes originales textuelles
X_final = X_encoded.drop(['nombre_cientifico', 'ubicacion_plantera', 'comuna'], axis=1)

# Normalisation des features numériques
scaler = StandardScaler()
numerical_features = ['altura_arbol', 'diametro_altura_pecho', 'ancho_acera', 'ratio_altura_diametro', 'volume_estimate']
X_final[numerical_features] = scaler.fit_transform(X_final[numerical_features])

In [5]:
# STRATÉGIE DE RÉÉCHANTILLONNAGE CORRIGÉE
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, 
    test_size=0.2,
    random_state=42, 
    stratify=y
)

print("=== STRATÉGIE DE RÉÉCHANTILLONNAGE ===")
print(f"Distribution originale: {Counter(y_train)}")

# SMOTE avec stratégie automatique qui respecte les contraintes
smote = SMOTE(
    sampling_strategy='auto',  # Ou utiliser un ratio plus conservateur
    random_state=42,
    k_neighbors=2  # Réduire pour les petites classes
)

X_resampled, y_resampled = smote.fit_resample(X_train, y_train)
print(f"Distribution après SMOTE: {Counter(y_resampled)}")
print()

=== STRATÉGIE DE RÉÉCHANTILLONNAGE ===
Distribution originale: Counter({2.0: 241816, 3.0: 28046, 1.0: 134})
Distribution après SMOTE: Counter({2.0: 241816, 3.0: 241816, 1.0: 241816})



In [6]:
# OPTIMISATION DES HYPERPARAMÈTRES AVEC CLASS WEIGHT
param_grid = {
    'n_estimators': [200],
    'max_depth': [15],
    'min_samples_split': [5, 10],
    'class_weight': ['balanced', 'balanced_subsample']
}

rf_model = RandomForestClassifier(random_state=42, n_jobs=-1)

grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    cv=3,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)

print("=== OPTIMISATION DES HYPERPARAMÈTRES ===")
grid_search.fit(X_resampled, y_resampled)
print(f"Meilleurs paramètres: {grid_search.best_params_}")
print()

best_rf_model = grid_search.best_estimator_

=== OPTIMISATION DES HYPERPARAMÈTRES ===
Fitting 3 folds for each of 4 candidates, totalling 12 fits
Meilleurs paramètres: {'class_weight': 'balanced', 'max_depth': 15, 'min_samples_split': 5, 'n_estimators': 200}



In [ ]:
# ÉVALUATION DU MODÈLE AMÉLIORÉ
y_pred = best_rf_model.predict(X_test)

print("=== RAPPORT DE CLASSIFICATION ===")
print(classification_report(y_test, y_pred))
print()

print("=== MATRICE DE CONFUSION ===")
print(confusion_matrix(y_test, y_pred))
print()

accuracy = accuracy_score(y_test, y_pred)
print(f"=== PRÉCISION GLOBALE: {accuracy:.3f} ===")
print()

# Importance des features
feature_importance = pd.DataFrame({
    'feature': X_final.columns,
    'importance': best_rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("=== IMPORTANCE DES VARIABLES ===")
print(feature_importance)

=== RAPPORT DE CLASSIFICATION AMÉLIORÉ ===
              precision    recall  f1-score   support

         1.0       0.02      0.24      0.04        33
         2.0       0.93      0.85      0.89     60454
         3.0       0.27      0.46      0.34      7012

    accuracy                           0.81     67499
   macro avg       0.41      0.52      0.42     67499
weighted avg       0.86      0.81      0.83     67499


=== MATRICE DE CONFUSION ===
[[    8    13    12]
 [  289 51230  8935]
 [  106  3680  3226]]

=== PRÉCISION GLOBALE: 0.807 ===

=== IMPORTANCE DES VARIABLES ===
                      feature  importance
0                altura_arbol    0.291655
4   nombre_cientifico_encoded    0.153255
3              nivel_plantera    0.105100
8              comuna_encoded    0.099399
2                 ancho_acera    0.099075
7             volume_estimate    0.093516
6       ratio_altura_diametro    0.085046
1       diametro_altura_pecho    0.064403
5  ubicacion_plantera_encoded    0.0

In [10]:
# APPROCHE ALTERNATIVE: UNDER-SAMPLING STRATÉGIQUE
from imblearn.under_sampling import RandomUnderSampler

# Under-sampling de la classe majoritaire seulement
rus = RandomUnderSampler(
    sampling_strategy='auto',  # Garder toutes les instances des classes minoritaires
    random_state=42
)

X_under, y_under = rus.fit_resample(X_train, y_train)

print("=== APPROCHE ALTERNATIVE: UNDER-SAMPLING + CLASS WEIGHT ===")
print(f"Distribution après under-sampling: {Counter(y_under)}")
print()

# Entraîner un modèle sur les données under-samplées
rf_under = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_under.fit(X_under, y_under)
y_pred_under = rf_under.predict(X_test)

print("=== PERFORMANCES AVEC UNDER-SAMPLING ===")
print(classification_report(y_test, y_pred_under))
print(f"F1-score weighted: {f1_score(y_test, y_pred_under, average='weighted'):.3f}")

=== APPROCHE ALTERNATIVE: UNDER-SAMPLING + CLASS WEIGHT ===
Distribution après under-sampling: Counter({1.0: 134, 2.0: 134, 3.0: 134})

=== PERFORMANCES AVEC UNDER-SAMPLING ===
              precision    recall  f1-score   support

         1.0       0.00      0.79      0.01        33
         2.0       0.93      0.55      0.69     60454
         3.0       0.13      0.45      0.20      7012

    accuracy                           0.54     67499
   macro avg       0.36      0.60      0.30     67499
weighted avg       0.85      0.54      0.64     67499

F1-score weighted: 0.643
